# Prepare and merge all data for Autoencoder

In [1]:
import pandas as pd
import re
from datetime import datetime

In [2]:
# Variables and Paths
ALL_DATA_CSV = "output/merged_data.csv"
# LATENT_FILE = "output/Experiments/LatentVectorAnalysis/train_latent_vectors_with_patno.csv"
LATENT_FILE = "output/Experiments/LatentVectorAnalysis/validation_latent_vectors_with_patno.csv"
DICOM_FILE = "data/csvData/dicom_metadata.csv"
# OUTPUT_FILE = "output/final_train_combined_ae_data.csv"
OUTPUT_FILE = "output/final_validation_combined_ae_data.csv"

In [3]:
# Load merged clinical data
df_merged = pd.read_csv(ALL_DATA_CSV)
print(f"Merged clinical data: {df_merged.shape}")

# Load latent vectors
df_latent = pd.read_csv(LATENT_FILE)
print(f"Latent vectors: {df_latent.shape}")

# Load DICOM metadata for scanner info
df_dicom = pd.read_csv(DICOM_FILE)
print(f"DICOM metadata: {df_dicom.shape}")

/tmp/ipykernel_1489/3150277792.py:2: DtypeWarning: Columns (30,40,41) have mixed types. Specify dtype option on import or set low_memory=False.
  df_merged = pd.read_csv(ALL_DATA_CSV)


Merged clinical data: (41816, 42)
Latent vectors: (586, 260)
DICOM metadata: (2986, 6)


In [4]:
df_latent_clean = df_latent.dropna(subset=['FilePath']).copy()
df_latent_clean.shape

(586, 260)

In [5]:
df_dicom.rename(columns={'file_path': 'FilePath'}, inplace=True)
df_dicom_latest = df_dicom.dropna(subset=['FilePath']).copy()
df_dicom_latest.shape

(2986, 6)

In [6]:
print("--- Latent DataFrame Path Example ---")
print(df_latent_clean['FilePath'].iloc[0])

print("\n--- DICOM DataFrame Path Example ---")
print(df_dicom_latest['FilePath'].iloc[0])

--- Latent DataFrame Path Example ---
data/Images/PPMI_Images_PD/3220/Reconstructed_DaTSCAN/2012-12-13_13_37_22.0/I418476/PPMI_3220_NM_Reconstructed_DaTSCAN_Br_20140402100218028_1_S196722_I418476.dcm

--- DICOM DataFrame Path Example ---
Images\PPMI_Images_PD\100001\Reconstructed_DaTSCAN\2020-09-09_17_07_33.0\I1452480\PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [7]:
# Function to normalize paths
def normalize_path(path_str):
    if pd.isna(path_str): return path_str
    
    # 1. Force forward slashes
    clean_p = path_str.replace('\\', '/')
    
    # 2. Remove 'data/' prefix if it exists to ensure matching
    if clean_p.startswith('data/'):
        clean_p = clean_p.replace('data/', '')
        
    # 3. Strip any leading/trailing whitespace
    return clean_p.strip()

# Apply to BOTH dataframes
df_latent_clean['Merge_Key'] = df_latent_clean['FilePath'].apply(normalize_path)
df_dicom_latest['Merge_Key'] = df_dicom_latest['FilePath'].apply(normalize_path)

# Check if they look the same now
print("New Key Latent:", df_latent_clean['Merge_Key'].iloc[0])
print("New Key DICOM: ", df_dicom_latest['Merge_Key'].iloc[0])

New Key Latent: Images/PPMI_Images_PD/3220/Reconstructed_DaTSCAN/2012-12-13_13_37_22.0/I418476/PPMI_3220_NM_Reconstructed_DaTSCAN_Br_20140402100218028_1_S196722_I418476.dcm
New Key DICOM:  Images/PPMI_Images_PD/100001/Reconstructed_DaTSCAN/2020-09-09_17_07_33.0/I1452480/PPMI_100001_NM_Reconstructed_DaTSCAN_Br_20210608102518754_1_S1028880_I1452480.dcm


In [8]:
df_latent_with_scanner = pd.merge(
    df_latent_clean,
    df_dicom_latest[['Merge_Key', 'Manufacturer', 'ManufacturerModelName']],
    on='Merge_Key',
    how='left'  # Keep all latent vectors, add scanner info
)
df_latent_with_scanner.shape

(586, 263)

In [9]:
df_latent_with_scanner.sample(5)

,latent_0,latent_1,latent_2,latent_3,latent_4,latent_5,latent_6,latent_7,latent_8,latent_9,...,latent_253,latent_254,latent_255,Label,FilePath,PATNO,Dataset,Merge_Key,Manufacturer,ManufacturerModelName
1,-3.252012,-27.842243,-3.508562,-3.273795,-0.610061,-4.472724,1.828977,0.691844,0.809003,-2.511206,...,-0.777064,1.540712,-0.345592,PD,data/Images/PPMI_Images_PD/3603/Reconstructed_...,3603,Validation,Images/PPMI_Images_PD/3603/Reconstructed_DaTSC...,PICKER,HERMES Workstation
476,1.947107,-20.769530,1.530318,-2.872526,-1.356585,3.944130,0.473960,-1.877050,0.345602,-2.226955,...,1.334232,-0.574055,3.077828,PD,data/Images/PPMI_Images_PD/3825/Reconstructed_...,3825,Validation,Images/PPMI_Images_PD/3825/Reconstructed_DaTSC...,SIEMENS NM,Encore2
309,-1.106624,-19.123184,1.830550,1.520757,-3.761359,1.586577,-0.997944,-2.608229,0.580435,-5.870439,...,2.376037,1.772646,-1.373636,PD,data/Images/PPMI_Images_PD/3502/Reconstructed_...,3502,Validation,Images/PPMI_Images_PD/3502/Reconstructed_DaTSC...,PICKER,HERMES Workstation
57,-0.614598,-11.549788,0.422681,-7.925971,-3.923482,11.985132,4.340293,8.782738,-2.855428,-6.769197,...,-1.192473,2.367432,-9.705495,PD,data/Images/PPMI_Images_PD/3387/Reconstructed_...,3387,Validation,Images/PPMI_Images_PD/3387/Reconstructed_DaTSC...,GE MEDICAL SYSTEMS,MILLENNIUM MG
475,0.007986,-21.693964,3.613629,-2.715718,-0.263692,3.607628,2.778621,-1.369618,2.249542,0.855362,...,2.568524,3.822928,4.656202,PD,data/Images/PPMI_Images_PD/3110/Reconstructed_...,3110,Validation,Images/PPMI_Images_PD/3110/Reconstructed_DaTSC...,SIEMENS NM,IP2


In [10]:
# 1. Robust Date Extraction (Finds YYYY-MM-DD anywhere in path)
def get_date_from_path(path_str):
    if pd.isna(path_str):
        return None
    
    # Regex to find pattern: 4 digits - 2 digits - 2 digits
    match = re.search(r'(\d{4}-\d{2}-\d{2})', str(path_str))
    if match:
        date_raw = match.group(1) # Extracts '2021-04-06'
        try:
            # Added datetime import requirement and better error handling
            return datetime.strptime(date_raw, '%Y-%m-%d').strftime('%m/%Y')
        except Exception:
            return None
    return None

# 2. Apply the fix
df_latent_with_scanner['DATSCAN_DATE'] = df_latent_with_scanner['FilePath'].apply(get_date_from_path)

# Verify we actually have dates now (Safe check)
dates_found = df_latent_with_scanner['DATSCAN_DATE'].dropna()
if not dates_found.empty:
    print("Latent Date Sample:", dates_found.iloc[0])
else:
    print("Warning: No dates could be extracted from FilePath. Check your regex or path format.")

# 3. Clean Clinical Data (df_merged)
df_merged['PATNO'] = pd.to_numeric(df_merged['PATNO'], errors='coerce').fillna(0).astype(int)
df_latent_with_scanner['PATNO'] = pd.to_numeric(df_latent_with_scanner['PATNO'], errors='coerce').fillna(0).astype(int)

# Convert clinical dates to strings, handle NaNs
df_merged['DATSCAN_DATE'] = pd.to_datetime(
    df_merged['DATSCAN_DATE'], 
    format='mixed', 
    errors='coerce'
).dt.strftime('%m/%Y')

# 4. Perform the Merge
df_combined = pd.merge(
    df_latent_with_scanner, 
    df_merged, 
    on=['PATNO', 'DATSCAN_DATE'], 
    how='inner'
)

print(f"Merge Shape: {df_combined.shape}")

Latent Date Sample: 12/2012
Merge Shape: (587, 304)


In [11]:
df_combined['DATSCAN_DATE'].sample(5)

86     09/2018
284    06/2014
309    11/2012
104    04/2013
477    07/2016
Name: DATSCAN_DATE, dtype: object

In [12]:
# Save the final merged dataset for the Autoencoder

df_combined.to_csv(OUTPUT_FILE, index=False)

print(f"Successfully saved merged data to: {OUTPUT_FILE}")
print(f"Final file contains {df_combined.shape[0]} rows and {df_combined.shape[1]} columns.")

Successfully saved merged data to: output/final_validation_combined_ae_data.csv
Final file contains 587 rows and 304 columns.
